In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.append(str(ROOT / 'src'))

from vehicle_telemetry.longitudinal import (
    BUCKETS,
    baseline_comparison,
    generate_report_markdown,
    latest_by_bucket,
    load_events,
    select_baseline,
    summarize_all_logs,
    write_outputs,
)

FA20DIT_ZONES = {
    'dam': [(0.0, 0.75, 'tab:red', 'concerning'), (0.75, 0.875, 'darkorange', 'investigate'), (0.875, 0.999, 'goldenrod', 'moderate'), (0.999, 1.05, 'tab:green', 'ideal')],
    'fbk': [(-8.0, -2.8, 'tab:red', 'danger'), (-2.8, -1.4, 'goldenrod', 'moderate'), (-1.4, 1.0, 'tab:green', 'ideal')],
    'fkl': [(-8.0, -2.8, 'tab:red', 'danger'), (-2.8, -1.4, 'goldenrod', 'moderate'), (-1.4, 1.0, 'tab:green', 'ideal')],
    'coolant_c': [(0.0, 98.0, 'tab:green', 'ideal'), (98.0, 105.0, 'goldenrod', 'moderate'), (105.0, 130.0, 'tab:red', 'danger')],
    'iat_c': [(0.0, 40.0, 'tab:green', 'ideal'), (40.0, 55.0, 'goldenrod', 'moderate'), (55.0, 100.0, 'tab:red', 'danger')],
    'trim_pct': [(-20.0, -10.0, 'tab:red', 'danger'), (-10.0, -5.0, 'goldenrod', 'moderate'), (-5.0, 5.0, 'tab:green', 'ideal'), (5.0, 8.0, 'goldenrod', 'moderate'), (8.0, 20.0, 'tab:red', 'danger')],
    'load_proxy': [(0.0, 10.0, 'tab:green', 'light'), (10.0, 20.0, 'goldenrod', 'moderate'), (20.0, 35.0, 'tab:orange', 'high'), (35.0, 80.0, 'tab:red', 'very high')],
}

events_path = ROOT / 'data' / 'events.csv'
raw_dir = ROOT / 'data' / 'raw'
output_dir = ROOT / 'outputs'

events_df = load_events(events_path)
summary_df, frames = summarize_all_logs(raw_dir, events_df)
if summary_df.empty:
    raise RuntimeError(f'No CSV logs found in {raw_dir}')
summary_df = summary_df.sort_values('log_datetime').reset_index(drop=True)
latest_map = latest_by_bucket(summary_df)
baseline_map = {bucket: select_baseline(summary_df, latest_map[bucket]) for bucket in BUCKETS}
report_md = generate_report_markdown(summary_df, latest_map, baseline_map, events_df)
output_paths = write_outputs(summary_df, report_md, output_dir)

def _num_series(frame, col):
    return pd.to_numeric(frame.get(col), errors='coerce') if col in frame.columns else pd.Series(dtype=float)

def _bucket_rows():
    rows = []
    for bucket in BUCKETS:
        latest = latest_map[bucket]
        baseline = baseline_map[bucket]
        rows.append({
            'bucket': f'{bucket[0]}/{bucket[1]}',
            'latest_log': None if latest is None else latest['filename'],
            'log_datetime': None if latest is None else latest['log_datetime'],
            'baseline_count': len(baseline),
            'baseline_confidence': 'limited' if len(baseline) < 3 else 'normal',
        })
    return pd.DataFrame(rows)

def _metadata_table(latest_row):
    return pd.DataFrame([{
        'filename': latest_row['filename'],
        'log_datetime': latest_row['log_datetime'],
        'log_type': latest_row['log_type'],
        'session_type': latest_row['session_type'],
        'era_id': latest_row['era_id'],
        'datetime_source': latest_row['log_datetime_source'],
        'recent_event_context': latest_row.get('recent_event_context', ''),
    }])

def _bucket_history(bucket):
    return summary_df[(summary_df['log_type'] == bucket[0]) & (summary_df['session_type'] == bucket[1])].sort_values('log_datetime').copy()

def _plot_metric_trend(bucket, metric, title, ylabel):
    hist = _bucket_history(bucket)
    if metric not in hist.columns:
        return
    vals = pd.to_numeric(hist[metric], errors='coerce')
    hist = hist.assign(_metric=vals).dropna(subset=['_metric'])
    if hist.empty:
        return
    labels = [f"{row.filename}\n{pd.to_datetime(row.log_datetime).strftime('%Y-%m-%d')}" for row in hist.itertuples()]
    x = np.arange(len(hist))
    fig, ax = plt.subplots(figsize=(max(8.5, len(hist) * 1.6), 2.8))
    ax.plot(x, hist['_metric'], marker='o', lw=1.1, color='black')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha='right')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Logs')
    plt.tight_layout()
    plt.show()

def _plot_trim_trend(bucket):
    hist = _bucket_history(bucket).copy()
    if hist.empty:
        return
    hist['mean_ltft_pct'] = pd.to_numeric(hist.get('mean_ltft_pct'), errors='coerce')
    hist['mean_stft_pct'] = pd.to_numeric(hist.get('mean_stft_pct'), errors='coerce')
    if hist[['mean_ltft_pct', 'mean_stft_pct']].dropna(how='all').empty:
        return
    labels = [f"{row.filename}\n{pd.to_datetime(row.log_datetime).strftime('%Y-%m-%d')}" for row in hist.itertuples()]
    x = np.arange(len(hist))
    fig, ax = plt.subplots(figsize=(max(8.5, len(hist) * 1.6), 2.8))
    if hist['mean_ltft_pct'].notna().any():
        ax.plot(x, hist['mean_ltft_pct'], marker='o', lw=1.0, label='mean LTFT', color='tab:brown')
    if hist['mean_stft_pct'].notna().any():
        ax.plot(x, hist['mean_stft_pct'], marker='o', lw=1.0, label='mean STFT', color='tab:purple')
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha='right')
    ax.set_title('Historical Fuel Trim Trend')
    ax.set_ylabel('Trim (%)')
    ax.set_xlabel('Logs')
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

def _latest_cobb_row():
    cobb = [row for bucket, row in latest_map.items() if bucket[0] == "cobb" and row is not None]
    if not cobb:
        return None
    return sorted(cobb, key=lambda r: r['log_datetime'])[-1]

def _latest_overall_row():
    return summary_df.sort_values('log_datetime').iloc[-1] if not summary_df.empty else None

def _risk_label(color):
    return color

def _normal_range(metric):
    ranges = {
        'DAM': '1.00 ideal; 0.875-<1.00 moderate; 0.75-<0.875 investigate; <0.75 concerning',
        'FBK': '> -1.4 deg ideal; -1.4 to -2.8 deg moderate; < -2.8 deg concerning',
        'FKL': '> -1.4 deg ideal; -1.4 to -2.8 deg investigate; < -2.8 deg concerning',
        'Mean LTFT': '-5% to +5% typical; +/-5% to +/-8% moderate; beyond +/-8% investigate',
        'Mean STFT': '-5% to +5% typical; +/-5% to +/-8% moderate; beyond +/-8% investigate',
        'Max IAT': '<40 C ideal; 40-55 C moderate; >55 C concerning',
        'Max Coolant': '<98 C ideal; 98-105 C moderate; >105 C concerning',
        'Engine Demand': 'Light to very high, based on load proxy, MAP, and high-load exposure',
        'Risk Under Load': 'Low if loaded operation stays thermally stable and knock/timing remain quiet',
    }
    return ranges.get(metric, 'n/a')

def _style_summary_table(df):
    if df.empty:
        return HTML('<p>No summary metrics available.</p>')

    color_map = {
        'green': '#dff2e1',
        'yellow': '#fff4cc',
        'red': '#f8d7da',
        'gray': '#eceff3',
    }

    display_df = df.copy()
    if 'summary_stat' in display_df.columns:
        display_df['summary_stat'] = display_df['summary_stat'].map(
            lambda v: 'n/a' if pd.isna(v) else (f'{v:.3f}' if isinstance(v, (int, float, np.floating)) else str(v))
        )

    headers = ''.join(
        f"<th style='text-align:left; padding:6px 8px; border-bottom:1px solid #d0d7de'>{col}</th>"
        for col in display_df.columns
    )
    rows = []
    for _, row in display_df.iterrows():
        cells = []
        for col in display_df.columns:
            value = row[col]
            style = "padding:6px 8px; text-align:left; border-bottom:1px solid #eef2f6;"
            if col == 'risk':
                bg = color_map.get(str(value).lower(), '#ffffff')
                style += f" background-color:{bg}; font-weight:600;"
            cells.append(f"<td style='{style}'>{value}</td>")
        rows.append('<tr>' + ''.join(cells) + '</tr>')

    table_html = (
        "<table style='border-collapse:collapse; width:100%; font-size:13px'>"
        + f"<thead><tr>{headers}</tr></thead>"
        + f"<tbody>{''.join(rows)}</tbody></table>"
    )
    return HTML(table_html)

def _dam_status(value):
    if pd.isna(value):
        return ('n/a', 'gray')
    if value < 0.75:
        return ('concerning', 'red')
    if value < 0.875:
        return ('investigate', 'yellow')
    if value < 1.0:
        return ('moderate', 'yellow')
    return ('ideal', 'green')

def _knock_status(value):
    if pd.isna(value):
        return ('n/a', 'gray')
    if value <= -2.8:
        return ('deep correction', 'red')
    if value <= -1.4:
        return ('meaningful correction', 'yellow')
    return ('quiet', 'green')

def _trim_status(value):
    if pd.isna(value):
        return ('n/a', 'gray')
    av = abs(value)
    if av > 8:
        return ('biased', 'red')
    if av > 5:
        return ('elevated', 'yellow')
    return ('stable', 'green')

def _iat_status(value):
    if pd.isna(value):
        return ('n/a', 'gray')
    if value > 55:
        return ('hot', 'red')
    if value > 40:
        return ('elevated', 'yellow')
    return ('normal', 'green')

def _coolant_status(value):
    if pd.isna(value):
        return ('n/a', 'gray')
    if value > 105:
        return ('hot', 'red')
    if value > 98:
        return ('elevated', 'yellow')
    return ('normal', 'green')

def _load_status(value):
    if pd.isna(value):
        return ('n/a', 'gray')
    if value >= 40:
        return ('very high', 'red')
    if value >= 25:
        return ('high', 'yellow')
    if value >= 15:
        return ('moderate', 'green')
    return ('light', 'green')

def _engine_demand_status(max_load, p95_load, high_load_pct, max_map):
    max_load = pd.to_numeric(pd.Series([max_load]), errors='coerce').iloc[0]
    p95_load = pd.to_numeric(pd.Series([p95_load]), errors='coerce').iloc[0]
    high_load_pct = pd.to_numeric(pd.Series([high_load_pct]), errors='coerce').iloc[0]
    max_map = pd.to_numeric(pd.Series([max_map]), errors='coerce').iloc[0]
    if pd.isna(max_load) and pd.isna(p95_load) and pd.isna(high_load_pct) and pd.isna(max_map):
        return ('n/a', 'gray')
    if ((pd.notna(max_load) and max_load >= 40) or (pd.notna(p95_load) and p95_load >= 28) or (pd.notna(high_load_pct) and high_load_pct >= 18) or (pd.notna(max_map) and max_map >= 240)):
        return ('very high', 'red')
    if ((pd.notna(max_load) and max_load >= 25) or (pd.notna(p95_load) and p95_load >= 18) or (pd.notna(high_load_pct) and high_load_pct >= 8) or (pd.notna(max_map) and max_map >= 210)):
        return ('high', 'yellow')
    if ((pd.notna(max_load) and max_load >= 15) or (pd.notna(p95_load) and p95_load >= 12) or (pd.notna(high_load_pct) and high_load_pct >= 3) or (pd.notna(max_map) and max_map >= 170)):
        return ('moderate', 'green')
    return ('light', 'green')

def _risk_under_load_status(row):
    min_dam = pd.to_numeric(pd.Series([row.get('min_dam')]), errors='coerce').iloc[0]
    min_fbk = pd.to_numeric(pd.Series([row.get('min_fbk')]), errors='coerce').iloc[0]
    min_fkl = pd.to_numeric(pd.Series([row.get('min_fkl')]), errors='coerce').iloc[0]
    fbk_count = pd.to_numeric(pd.Series([row.get('meaningful_fbk_count')]), errors='coerce').iloc[0]
    fkl_count = pd.to_numeric(pd.Series([row.get('meaningful_fkl_count')]), errors='coerce').iloc[0]
    max_iat = pd.to_numeric(pd.Series([row.get('max_iat_c')]), errors='coerce').iloc[0]
    max_coolant = pd.to_numeric(pd.Series([row.get('max_coolant_c')]), errors='coerce').iloc[0]
    mean_ltft = pd.to_numeric(pd.Series([row.get('mean_ltft_pct')]), errors='coerce').iloc[0]
    mean_stft = pd.to_numeric(pd.Series([row.get('mean_stft_pct')]), errors='coerce').iloc[0]
    high_load_pct = pd.to_numeric(pd.Series([row.get('high_load_pct')]), errors='coerce').iloc[0]
    max_load = pd.to_numeric(pd.Series([row.get('max_load_proxy')]), errors='coerce').iloc[0]

    loaded = ((pd.notna(high_load_pct) and high_load_pct >= 3) or (pd.notna(max_load) and max_load >= 15))
    issue_count = 0.0
    severe = False

    if pd.notna(min_dam) and min_dam < 0.875:
        issue_count += 1
        severe = severe or min_dam < 0.75
    if pd.notna(min_fbk) and min_fbk <= -2.8:
        issue_count += 1
        severe = True
    elif pd.notna(min_fbk) and min_fbk <= -2.0:
        issue_count += 1
    if pd.notna(min_fkl) and min_fkl <= -2.8:
        issue_count += 1
        severe = True
    elif pd.notna(min_fkl) and min_fkl <= -1.4:
        issue_count += 1
    if pd.notna(fbk_count) and fbk_count >= 2:
        issue_count += 1
    if pd.notna(fkl_count) and fkl_count >= 2:
        issue_count += 1
    if pd.notna(max_iat) and max_iat > 55:
        issue_count += 1
        severe = True
    elif pd.notna(max_iat) and max_iat > 40:
        issue_count += 0.5
    if pd.notna(max_coolant) and max_coolant > 105:
        issue_count += 1
        severe = True
    elif pd.notna(max_coolant) and max_coolant > 98:
        issue_count += 0.5
    if pd.notna(mean_ltft) and abs(mean_ltft) > 8:
        issue_count += 1
    elif pd.notna(mean_ltft) and abs(mean_ltft) > 5:
        issue_count += 0.5
    if pd.notna(mean_stft) and abs(mean_stft) > 8:
        issue_count += 1
    elif pd.notna(mean_stft) and abs(mean_stft) > 5:
        issue_count += 0.5

    if not loaded and issue_count == 0:
        return ('low', 'green')
    if severe or (loaded and issue_count >= 3):
        return ('high', 'red')
    if issue_count >= 1.5:
        return ('investigate', 'yellow')
    if issue_count > 0:
        return ('watch', 'yellow')
    return ('low', 'green')

def _summary_bullets_and_table():
    overall = _latest_overall_row()
    cobb = _latest_cobb_row()
    if overall is None:
        return ['No logs available.'], pd.DataFrame()

    ref = cobb if cobb is not None else overall
    bullets = [f"Latest overall log: {overall['filename']} ({overall['log_type']} / {overall['session_type']})."]
    metrics = []

    if ref is cobb:
        min_dam = pd.to_numeric(pd.Series([ref.get('min_dam')]), errors='coerce').iloc[0]
        final_dam = pd.to_numeric(pd.Series([ref.get('final_dam')]), errors='coerce').iloc[0]
        min_fbk = pd.to_numeric(pd.Series([ref.get('min_fbk')]), errors='coerce').iloc[0]
        min_fkl = pd.to_numeric(pd.Series([ref.get('min_fkl')]), errors='coerce').iloc[0]
        dam_note, dam_risk = _dam_status(min_dam)
        metrics.append({'metric': 'DAM', 'summary_stat': min_dam, 'status': dam_note, 'risk': dam_risk, 'normal_range': _normal_range('DAM')})
        fbk_note, fbk_risk = _knock_status(min_fbk)
        metrics.append({'metric': 'FBK', 'summary_stat': min_fbk, 'status': fbk_note, 'risk': fbk_risk, 'normal_range': _normal_range('FBK')})
        fkl_note, fkl_risk = _knock_status(min_fkl)
        metrics.append({'metric': 'FKL', 'summary_stat': min_fkl, 'status': fkl_note, 'risk': fkl_risk, 'normal_range': _normal_range('FKL')})
        if dam_risk != 'green':
            bullets.append(f"DAM flagged: minimum DAM was {min_dam:.3f} and final DAM was {final_dam:.3f}.")
        if fbk_risk != 'green' or fkl_risk != 'green':
            bullets.append('Knock correction flagged: meaningful FBK/FKL was observed in the latest detailed log.')
        if dam_risk == 'green' and fbk_risk == 'green' and fkl_risk == 'green':
            bullets.append('No major Cobb knock/timing anomalies were flagged.')

    mean_ltft = pd.to_numeric(pd.Series([overall.get('mean_ltft_pct')]), errors='coerce').iloc[0]
    mean_stft = pd.to_numeric(pd.Series([overall.get('mean_stft_pct')]), errors='coerce').iloc[0]
    max_iat = pd.to_numeric(pd.Series([overall.get('max_iat_c')]), errors='coerce').iloc[0]
    max_coolant = pd.to_numeric(pd.Series([overall.get('max_coolant_c')]), errors='coerce').iloc[0]
    max_load = pd.to_numeric(pd.Series([overall.get('max_load_proxy')]), errors='coerce').iloc[0]
    p95_load = pd.to_numeric(pd.Series([overall.get('p95_load_proxy')]), errors='coerce').iloc[0]
    high_load_pct = pd.to_numeric(pd.Series([overall.get('high_load_pct')]), errors='coerce').iloc[0]
    max_map = pd.to_numeric(pd.Series([overall.get('max_map_kpa')]), errors='coerce').iloc[0]

    ltft_note, ltft_risk = _trim_status(mean_ltft)
    stft_note, stft_risk = _trim_status(mean_stft)
    iat_note, iat_risk = _iat_status(max_iat)
    coolant_note, coolant_risk = _coolant_status(max_coolant)
    demand_note, demand_risk = _engine_demand_status(max_load, p95_load, high_load_pct, max_map)
    load_risk_note, load_risk_color = _risk_under_load_status(ref)

    demand_stat = 'n/a'
    if pd.notna(max_load) or pd.notna(p95_load) or pd.notna(high_load_pct):
        demand_stat = f"max load {max_load:.2f}; p95 {p95_load:.2f}; high-load {high_load_pct:.1f}%"
    risk_stat = f"DAM {ref.get('min_dam', np.nan)}; FBK {ref.get('min_fbk', np.nan)}; FKL {ref.get('min_fkl', np.nan)}; IAT {max_iat:.1f}C"

    metrics.extend([
        {'metric': 'Mean LTFT', 'summary_stat': mean_ltft, 'status': ltft_note, 'risk': ltft_risk, 'normal_range': _normal_range('Mean LTFT')},
        {'metric': 'Mean STFT', 'summary_stat': mean_stft, 'status': stft_note, 'risk': stft_risk, 'normal_range': _normal_range('Mean STFT')},
        {'metric': 'Max IAT', 'summary_stat': max_iat, 'status': iat_note, 'risk': iat_risk, 'normal_range': _normal_range('Max IAT')},
        {'metric': 'Max Coolant', 'summary_stat': max_coolant, 'status': coolant_note, 'risk': coolant_risk, 'normal_range': _normal_range('Max Coolant')},
        {'metric': 'Engine Demand', 'summary_stat': demand_stat, 'status': demand_note, 'risk': demand_risk, 'normal_range': _normal_range('Engine Demand')},
        {'metric': 'Risk Under Load', 'summary_stat': risk_stat, 'status': load_risk_note, 'risk': load_risk_color, 'normal_range': _normal_range('Risk Under Load')},
    ])

    if ltft_risk == 'red' or stft_risk == 'red':
        bullets.append('Fuel trims flagged: correction magnitude is outside the ideal band and worth investigating.')
    elif ltft_risk == 'yellow' or stft_risk == 'yellow':
        bullets.append('Fuel trims are mildly elevated and worth monitoring.')
    else:
        bullets.append('Fuel trims look stable in the latest overall log.')

    if iat_risk == 'red' or coolant_risk == 'red':
        bullets.append('Thermal behavior flagged: IAT or coolant entered a high-risk range.')
    elif iat_risk == 'yellow' or coolant_risk == 'yellow':
        bullets.append('Thermal behavior was elevated but not extreme.')
    else:
        bullets.append('Thermal behavior stayed in a normal range.')

    if demand_risk == 'red':
        bullets.append('Engine demand was very high in the latest session, but that is being treated as operating context rather than a direct safety issue by itself.')
    elif demand_note in ('high', 'moderate'):
        bullets.append(f'Engine demand was {demand_note} in the latest session.')
    else:
        bullets.append('The latest session was mostly light-load.')

    if load_risk_color == 'red':
        bullets.append('Risk under load was flagged high because multiple warning signals overlapped during loaded operation.')
    elif load_risk_color == 'yellow':
        bullets.append(f'Risk under load is worth review: status is {load_risk_note}.')
    else:
        bullets.append('No major anomalies were flagged during meaningful loaded operation.')

    return bullets, pd.DataFrame(metrics)

def _baseline_note(baseline):
    if len(baseline) < 3:
        return 'Baseline comparison uses up to 6 prior matching logs. Confidence is limited because fewer than 3 were available.'
    return 'Baseline comparison uses up to 6 prior matching logs. Confidence is normal.'

def _display_metric_section(title, explanation, interpretation):
    display(Markdown(f"### {title}"))
    display(Markdown(explanation))
    display(Markdown(f"**Interpretation:** {interpretation}"))

def _add_zone_bands(ax, bands):
    for low, high, color, label in bands:
        ax.axhspan(low, high, color=color, alpha=0.14, zorder=0)
        ax.text(0.995, (low + high) / 2.0, label, color=color, fontsize=8, ha='right', va='center', transform=ax.get_yaxis_transform())

def _plot_dam(frame):
    t = _num_series(frame, "time_s")
    dam = _num_series(frame, "dam")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    _add_zone_bands(ax, FA20DIT_ZONES['dam'])
    if dam.notna().any():
        ax.plot(t, dam, lw=1.2, label='DAM', color='black')
        ax.axhline(1.0, color='red', ls='--', lw=1.0, label='Reference 1.0')
        ax.legend(loc='best')
    else:
        ax.text(0.5, 0.5, "No DAM data", ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
    ax.set_title('DAM vs Time')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('DAM')
    plt.tight_layout()
    plt.show()

def _plot_fbk(frame):
    t = _num_series(frame, "time_s")
    fbk = _num_series(frame, "fbk")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    _add_zone_bands(ax, FA20DIT_ZONES['fbk'])
    if fbk.notna().any():
        ax.plot(t, fbk, lw=1.1, label='FBK', color='black')
        ax.axhline(-1.4, color='gray', ls='--', lw=0.8, label='-1.4 reference')
        ax.axhline(-2.8, color='orange', ls='--', lw=1.0, label='-2.8 reference')
        ax.legend(loc='best')
    else:
        ax.text(0.5, 0.5, "No FBK data", ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
    ax.set_title('FBK vs Time')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('FBK (deg)')
    plt.tight_layout()
    plt.show()

def _plot_fkl(frame):
    t = _num_series(frame, "time_s")
    fkl = _num_series(frame, "fkl")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    _add_zone_bands(ax, FA20DIT_ZONES['fkl'])
    if fkl.notna().any():
        ax.plot(t, fkl, lw=1.1, label='FKL', color='black')
        ax.axhline(-1.4, color='gray', ls='--', lw=0.8, label='-1.4 reference')
        ax.axhline(-2.8, color='orange', ls='--', lw=1.0, label='-2.8 reference')
        ax.legend(loc='best')
    else:
        ax.text(0.5, 0.5, "No FKL data", ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
    ax.set_title('FKL vs Time')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('FKL (deg)')
    plt.tight_layout()
    plt.show()

def _plot_stress(frame):
    t = _num_series(frame, "time_s")
    load = _num_series(frame, "load_proxy")
    map_kpa = _num_series(frame, "map_kpa")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    _add_zone_bands(ax, FA20DIT_ZONES['load_proxy'])
    if load.notna().any():
        ax.plot(t, load, lw=1.1, label='load_proxy', color='black')
    if map_kpa.notna().any():
        ax2 = ax.twinx()
        ax2.plot(t, map_kpa, lw=1.0, color='tab:red', alpha=0.7, label='MAP')
        ax2.set_ylabel('MAP (kPa)')
    ax.set_title('Load Proxy and MAP vs Time')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Load Proxy')
    plt.tight_layout()
    plt.show()

def _plot_rpm_context(frame):
    rpm = _num_series(frame, 'rpm')
    maf = _num_series(frame, 'maf_gps')
    map_kpa = _num_series(frame, 'map_kpa')
    fbk = _num_series(frame, 'fbk')
    fkl = _num_series(frame, 'fkl')

    fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=False)

    maf_df = pd.DataFrame({'rpm': rpm, 'maf': maf}).dropna()
    if not maf_df.empty:
        h0 = axes[0].hist2d(maf_df['rpm'], maf_df['maf'], bins=[30, 30], cmap='Blues', cmin=1)
        plt.colorbar(h0[3], ax=axes[0], label='Sample Density')
    else:
        axes[0].text(0.5, 0.5, 'No RPM/MAF data', ha='center', va='center', transform=axes[0].transAxes)
        axes[0].set_axis_off()
    axes[0].set_title('MAF vs RPM Density')
    axes[0].set_ylabel('MAF (g/s)')
    axes[0].set_xlabel('RPM')

    map_df = pd.DataFrame({'rpm': rpm, 'map': map_kpa}).dropna()
    if not map_df.empty:
        h1 = axes[1].hist2d(map_df['rpm'], map_df['map'], bins=[30, 30], cmap='Reds', cmin=1)
        plt.colorbar(h1[3], ax=axes[1], label='Sample Density')
    else:
        axes[1].text(0.5, 0.5, 'No RPM/MAP data', ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_axis_off()
    axes[1].set_title('MAP vs RPM Density')
    axes[1].set_ylabel('MAP (kPa)')
    axes[1].set_xlabel('RPM')

    rpm_bins = np.arange(0, max(8000, float(rpm.dropna().max()) + 500 if rpm.notna().any() else 8000), 500)
    fbk_counts = pd.cut(rpm[fbk <= -1.4], bins=rpm_bins).value_counts(sort=False)
    fkl_counts = pd.cut(rpm[fkl <= -1.4], bins=rpm_bins).value_counts(sort=False)
    if fbk_counts.sum() > 0 or fkl_counts.sum() > 0:
        centers = [interval.mid for interval in fbk_counts.index]
        axes[2].bar([c - 70 for c in centers], fbk_counts.values, width=120, label='FBK <= -1.4', color='tab:purple', alpha=0.75)
        axes[2].bar([c + 70 for c in centers], fkl_counts.values, width=120, label='FKL <= -1.4', color='tab:orange', alpha=0.75)
        axes[2].legend(loc='best')
    else:
        axes[2].text(0.5, 0.5, 'No meaningful RPM/knock events', ha='center', va='center', transform=axes[2].transAxes)
        axes[2].set_axis_off()
    axes[2].set_title('Meaningful Knock Events by RPM Band')
    axes[2].set_ylabel('Event Count')
    axes[2].set_xlabel('RPM')

    plt.tight_layout()
    plt.show()

def _plot_coolant(frame):
    t = _num_series(frame, "time_s")
    coolant = _num_series(frame, "coolant_c")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    _add_zone_bands(ax, FA20DIT_ZONES['coolant_c'])
    if coolant.notna().any():
        ax.plot(t, coolant, lw=1.0, label='Coolant C', color='tab:blue')
        ax.legend(loc='best')
    else:
        ax.text(0.5, 0.5, "No coolant data", ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
    ax.set_title('Coolant vs Time')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Coolant (C)')
    plt.tight_layout()
    plt.show()

def _plot_iat(frame):
    t = _num_series(frame, "time_s")
    iat = _num_series(frame, "iat_c")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    _add_zone_bands(ax, FA20DIT_ZONES['iat_c'])
    if iat.notna().any():
        ax.plot(t, iat, lw=1.0, label='IAT C', color='tab:orange')
        ax.legend(loc='best')
    else:
        ax.text(0.5, 0.5, "No IAT data", ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
    ax.set_title('IAT vs Time')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('IAT (C)')
    plt.tight_layout()
    plt.show()

def _plot_trims(frame):
    t = _num_series(frame, "time_s")
    stft = _num_series(frame, "stft_pct")
    ltft = _num_series(frame, "ltft_pct")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    _add_zone_bands(ax, FA20DIT_ZONES['trim_pct'])
    if stft.notna().any(): ax.plot(t, stft, lw=1.0, label='STFT', color='tab:purple')
    if ltft.notna().any(): ax.plot(t, ltft, lw=1.0, label='LTFT', color='tab:brown')
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    if stft.notna().any() or ltft.notna().any(): ax.legend(loc='best')
    ax.set_title('Fuel Trims vs Time')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Trim (%)')
    plt.tight_layout()
    plt.show()

def _interpret_dam(row):
    min_dam = pd.to_numeric(pd.Series([row.get('min_dam')]), errors='coerce').iloc[0]
    final_dam = pd.to_numeric(pd.Series([row.get('final_dam')]), errors='coerce').iloc[0]
    if pd.isna(min_dam):
        return 'No DAM data available.'
    if final_dam < 0.75:
        return 'DAM finished in the concerning range, which indicates materially reduced ECU confidence.'
    if min_dam < 0.75:
        return 'DAM entered the concerning range during the session, even if it later recovered.'
    if min_dam < 0.875:
        return 'DAM entered the investigate range, which suggests the log deserves closer review.'
    if min_dam < 1.0:
        return 'DAM entered the moderate range, indicating some reduction in timing confidence.'
    return 'DAM stayed in the ideal range throughout the session.'

def _interpret_fbk(row):
    min_fbk = pd.to_numeric(pd.Series([row.get('min_fbk')]), errors='coerce').iloc[0]
    if pd.isna(min_fbk):
        return 'No FBK data available.'
    if min_fbk <= -2.8:
        return 'FBK reached the FA20DIT danger zone, meaning deeper instantaneous timing pull occurred.'
    if min_fbk <= -1.4:
        return 'FBK reached the moderate zone, meaning some meaningful instantaneous correction occurred.'
    return 'FBK stayed in the ideal zone; only very small or no instantaneous correction was present.'

def _interpret_fkl(row):
    min_fkl = pd.to_numeric(pd.Series([row.get('min_fkl')]), errors='coerce').iloc[0]
    if pd.isna(min_fkl):
        return 'No FKL data available.'
    if min_fkl <= -2.8:
        return 'FKL entered the FA20DIT danger zone, suggesting stronger learned correction in repeated operating cells.'
    if min_fkl <= -1.4:
        return 'FKL entered the moderate zone, indicating mild-to-moderate learned correction.'
    return 'FKL stayed in the ideal zone; no meaningful learned correction was recorded.'

def _interpret_risk(row):
    high_load = pd.to_numeric(pd.Series([row.get('high_load_pct')]), errors='coerce').iloc[0]
    max_load = pd.to_numeric(pd.Series([row.get('max_load_proxy')]), errors='coerce').iloc[0]
    if pd.isna(high_load) or pd.isna(max_load):
        return 'Load data was not available.'
    if max_load >= 35 or high_load >= 20:
        return 'Engine load reached the very-high range and reflects hard-use operation, but this is descriptive rather than automatically unsafe.'
    if max_load >= 20 or high_load >= 12:
        return 'Engine load reached the high range and reflects meaningful boost/load use.'
    if max_load >= 10 or high_load >= 8:
        return 'Engine load reached the moderate range and reflects ordinary spirited driving.'
    return 'Engine load stayed in the light range and looks like easy cruising.'

def _interpret_coolant(row):
    max_coolant = pd.to_numeric(pd.Series([row.get('max_coolant_c')]), errors='coerce').iloc[0]
    if pd.isna(max_coolant):
        return 'Coolant data was not available.'
    if max_coolant > 105:
        return 'Coolant entered the FA20DIT danger zone and indicates hotter engine operation than desired.'
    if max_coolant > 98:
        return 'Coolant entered the moderate zone but did not reach an overheating range.'
    return 'Coolant stayed in the ideal FA20DIT operating band.'

def _interpret_iat(row):
    max_iat = pd.to_numeric(pd.Series([row.get('max_iat_c')]), errors='coerce').iloc[0]
    if pd.isna(max_iat):
        return 'IAT data was not available.'
    if max_iat > 55:
        return 'IAT entered the FA20DIT danger zone, which suggests notable heat soak or hot ambient/load conditions.'
    if max_iat > 40:
        return 'IAT entered the moderate zone but did not reach a severe level.'
    return 'IAT stayed in the ideal FA20DIT range for this dataset.'

def _interpret_trims(row):
    mean_ltft = pd.to_numeric(pd.Series([row.get('mean_ltft_pct')]), errors='coerce').iloc[0]
    mean_stft = pd.to_numeric(pd.Series([row.get('mean_stft_pct')]), errors='coerce').iloc[0]
    if pd.isna(mean_ltft) and pd.isna(mean_stft):
        return 'Fuel trim data was not available.'
    max_abs = np.nanmax([abs(mean_ltft) if pd.notna(mean_ltft) else np.nan, abs(mean_stft) if pd.notna(mean_stft) else np.nan])
    if max_abs > 8:
        return 'Fuel trims are in the FA20DIT danger zone and show a strong fueling bias worth investigating.'
    if max_abs > 5:
        return 'Fuel trims are in the moderate zone and show some bias, but not an extreme one.'
    return 'Fuel trims stayed in the ideal zone and look broadly stable.'

def _display_cobb_bucket(bucket, latest_row, baseline, frame):
    display(_metadata_table(latest_row))
    display(baseline_comparison(latest_row, baseline))
    display(Markdown(f"**{_baseline_note(baseline)}**"))
    _display_metric_section(
        "DAM Analysis",
        "DAM (Dynamic Advance Multiplier) reflects ECU confidence in safe ignition advance on the FA20DIT. A DAM of 1.00 is the target full-confidence state, meaning the ECU is willing to carry its full learned timing authority. Drops below 1.00 indicate reduced confidence, with progressively lower values meaning the ECU has become more conservative about knock risk.",
        _interpret_dam(latest_row),
    )
    _plot_dam(frame)
    _plot_metric_trend(bucket, 'min_dam', 'Historical Min DAM', 'Min DAM')
    _display_metric_section(
        "FBK Analysis",
        "FBK (Feedback Knock) is immediate timing pull when the FA20DIT ECU reacts to knock in the moment. It is the fast, short-term correction channel, so brief dips can happen without becoming a long-term issue. Deeper or repeated negative values matter more because they show the ECU actively removing timing under current load and RPM conditions.",
        _interpret_fbk(latest_row),
    )
    _plot_fbk(frame)
    _plot_metric_trend(bucket, 'min_fbk', 'Historical Min FBK', 'Min FBK (deg)')
    _display_metric_section(
        "FKL Analysis",
        "FKL (Fine Knock Learn) is retained learned correction in repeated RPM/load cells on the FA20DIT. Unlike FBK, which is immediate, FKL shows where the ECU has learned that certain operating regions need ongoing correction. That makes it useful for identifying persistent problem zones rather than one-off transients.",
        _interpret_fkl(latest_row),
    )
    _plot_fkl(frame)
    _plot_metric_trend(bucket, 'min_fkl', 'Historical Min FKL', 'Min FKL (deg)')
    _display_metric_section(
        "Engine Risk",
        "Load proxy is calculated as maf_gps / rpm * 1000 and serves as a simple airflow-per-revolution demand indicator. It is useful for describing how hard the engine was being asked to work, but it is not a direct safety limit by itself. The load bands in this report are descriptive only and should be interpreted together with MAP, IAT, DAM, and knock correction.",
        _interpret_risk(latest_row),
    )
    _plot_stress(frame)
    _plot_metric_trend(bucket, 'max_load_proxy', 'Historical Max Load Proxy', 'Max Load Proxy')
    _display_metric_section(
        "RPM Context",
        "These RPM-relative views show how airflow, manifold pressure, and meaningful knock activity are distributed across the rev range. The density plots make it easier to see where the engine spent most of its operating time, rather than over-emphasizing isolated points. The knock histogram is useful for spotting RPM bands where correction clusters repeatedly.",
        "Use this section to see whether any knock activity or airflow limitations are concentrated in a particular RPM region rather than spread evenly through the pull.",
    )
    _plot_rpm_context(frame)
    _display_metric_section(
        "Coolant Analysis",
        "Coolant temperature reflects the bulk operating temperature of the FA20DIT engine. It tells you how much thermal load the cooling system and engine block were carrying over the drive. Sustained elevation matters because coolant heat is slower-moving and more representative of overall engine thermal stress than transient intake heat.",
        _interpret_coolant(latest_row),
    )
    _plot_coolant(frame)
    _plot_metric_trend(bucket, 'max_coolant_c', 'Historical Max Coolant', 'Max Coolant (C)')
    _display_metric_section(
        "IAT Analysis",
        "Intake air temperature reflects how hot the intake charge became before combustion. On a turbo FA20DIT this matters because hotter intake charge generally reduces knock margin and can change how aggressively the ECU can hold timing. IAT also reacts faster than coolant, so it is useful for spotting heat soak and repeated boost exposure.",
        _interpret_iat(latest_row),
    )
    _plot_iat(frame)
    _plot_metric_trend(bucket, 'max_iat_c', 'Historical Max IAT', 'Max IAT (C)')
    _display_metric_section(
        "Fuel Trim Analysis",
        "STFT and LTFT show how much fueling correction the ECU needed relative to its base model. STFT reflects the short-term real-time correction, while LTFT reflects the slower learned bias. Together they help show whether the FA20DIT is fueling cleanly or leaning on correction to stay on target.",
        _interpret_trims(latest_row),
    )
    _plot_trims(frame)
    _plot_trim_trend(bucket)

def _display_obd_bucket(bucket, latest_row, baseline, frame):
    display(_metadata_table(latest_row))
    display(baseline_comparison(latest_row, baseline))
    display(Markdown(f"**{_baseline_note(baseline)}**"))
    _display_metric_section(
        "Engine Risk",
        "OBD logs do not include Subaru-specific DAM, FBK, or FKL channels here, so risk has to be inferred from load_proxy, MAP, MAF, and time spent at higher demand. That means this section is best used to understand how hard the engine was working, not to make a direct knock-confidence judgment. It is still useful for separating easy cruising from more boost-heavy or airflow-heavy sessions.",
        _interpret_risk(latest_row),
    )
    _plot_stress(frame)
    _plot_metric_trend(bucket, 'max_load_proxy', 'Historical Max Load Proxy', 'Max Load Proxy')
    _display_metric_section(
        "RPM Context",
        "These RPM-relative views show how airflow, manifold pressure, and meaningful knock activity are distributed across the rev range. The density plots make it easier to see where the engine spent most of its operating time, rather than over-emphasizing isolated points. The knock histogram is useful for spotting RPM bands where correction clusters repeatedly.",
        "Use this section to see whether any knock activity or airflow limitations are concentrated in a particular RPM region rather than spread evenly through the pull.",
    )
    _plot_rpm_context(frame)
    _display_metric_section(
        "Coolant Analysis",
        "OBD coolant temperature shows the engine's bulk operating temperature even when Subaru knock channels are unavailable. It is one of the better general health indicators in reduced-detail logs because it captures sustained thermal load. If coolant trends higher than expected for similar sessions, that can be a sign to inspect operating conditions or cooling performance.",
        _interpret_coolant(latest_row),
    )
    _plot_coolant(frame)
    _plot_metric_trend(bucket, 'max_coolant_c', 'Historical Max Coolant', 'Max Coolant (C)')
    _display_metric_section(
        "IAT Analysis",
        "OBD intake air temperature shows how hot the intake charge became during the drive. On the FA20DIT, elevated IAT reduces knock margin even if you cannot see DAM or knock correction directly in the OBD log. That makes it an important context metric for judging whether the engine was operating in a heat-soaked state.",
        _interpret_iat(latest_row),
    )
    _plot_iat(frame)
    _plot_metric_trend(bucket, 'max_iat_c', 'Historical Max IAT', 'Max IAT (C)')
    _display_metric_section(
        "Fuel Trim Analysis",
        "OBD short- and long-term fuel trims show whether the ECU had to add or remove fuel over the drive to stay on target. STFT captures immediate correction and LTFT captures learned bias over time. In reduced-detail logs, trims are one of the most useful indicators for whether fueling behavior remains centered or has started to drift.",
        _interpret_trims(latest_row),
    )
    _plot_trims(frame)
    _plot_trim_trend(bucket)

summary_df.head()


In [ ]:
display(Markdown('## Summary'))
summary_bullets, summary_metric_df = _summary_bullets_and_table()
bullet_md = '\n'.join([f'- {item}' for item in summary_bullets])
display(Markdown(bullet_md))
display(_style_summary_table(summary_metric_df))
display(_bucket_rows())
print('Outputs written:')
for key, value in output_paths.items():
    print(f'- {key}: {value}')


In [ ]:
display(Markdown('## Cobb Data'))
found = False
for bucket in [('cobb', 'cruising'), ('cobb', 'racing')]:
    latest = latest_map[bucket]
    if latest is None:
        continue
    found = True
    display(Markdown(f'## {bucket[0].upper()} / {bucket[1]}'))
    _display_cobb_bucket(bucket, latest, baseline_map[bucket], frames[latest['log_id']])
if not found:
    print('No Cobb logs found.')


In [ ]:
display(Markdown('## OBD Data'))
found = False
for bucket in [('obd', 'cruising'), ('obd', 'racing')]:
    latest = latest_map[bucket]
    if latest is None:
        continue
    found = True
    display(Markdown(f'## {bucket[0].upper()} / {bucket[1]}'))
    _display_obd_bucket(bucket, latest, baseline_map[bucket], frames[latest['log_id']])
if not found:
    print('No OBD logs found.')


In [ ]:
display(Markdown('## Metadata'))
display(summary_df[['filename','log_datetime','log_datetime_source','log_type','session_type','era_id','recent_event_context']])
baseline_rows = []
for bucket in BUCKETS:
    baseline = baseline_map[bucket]
    baseline_rows.append({
        'bucket': f'{bucket[0]}/{bucket[1]}',
        'baseline_count': len(baseline),
        'baseline_files': ', '.join(baseline['filename'].astype(str).tolist()) if not baseline.empty else '',
    })
display(pd.DataFrame(baseline_rows))
if not events_df.empty:
    display(events_df)
else:
    print('No events loaded from data/events.csv')
